# 3. Biang kerok: bahan mana yang bikin modal mie ayam naik?

Lanjutan notebook 2. Di sana kita tahu modal bahan (tanpa mie) naik sekitar **30,8%** sejak Jan 2021. Sekarang kita bongkar: **siapa yang paling bertanggung jawab?**

Ada dua pertanyaan yang kelihatan mirip tapi beda:
1. Bahan mana yang paling **besar porsinya** di modal? (soal ukuran)
2. Bahan mana yang paling **menyumbang kenaikan**? (soal perubahan)

> Terigu (mie) masih belum masuk karena belum ada di data. Jadi ini biang kerok di antara 4 bahan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

DATA = '/kaggle/input/datasets/samuelgeraldo/bedah-mie-ayam-pakai-data-bukan-sumpit/harga_pangan_bersih.csv'
df = pd.read_csv(DATA)
df['bulan'] = pd.to_datetime(df['bulan'])
pivot = df.pivot(index='bulan', columns='komoditas', values='harga_rp')
pivot = pivot.interpolate(method='linear', limit_direction='both')  # sama seperti notebook 2

RESEP = {'ayam_g': 60, 'bawang_merah_g': 10, 'bawang_putih_g': 5, 'minyak_ml': 20}
minyak = pivot[['Minyak Goreng Kemasan Bermerk 1', 'Minyak Goreng Kemasan Bermerk 2']].mean(axis=1)

biaya = pd.DataFrame(index=pivot.index)
biaya['ayam']         = pivot['Daging Ayam Ras Segar'] * RESEP['ayam_g'] / 1000
biaya['bawang_merah'] = pivot['Bawang Merah Ukuran Sedang'] * RESEP['bawang_merah_g'] / 1000
biaya['bawang_putih'] = pivot['Bawang Putih Ukuran Sedang'] * RESEP['bawang_putih_g'] / 1000
biaya['minyak']       = minyak * RESEP['minyak_ml'] / 1000
biaya = biaya[biaya.index < '2026-10-01']   # Okt 2026 belum utuh
BAHAN = ['ayam', 'bawang_merah', 'bawang_putih', 'minyak']
biaya['total'] = biaya[BAHAN].sum(axis=1)
biaya.round(0).tail(3)

## Langkah 1: Porsi tiap bahan di modal

Pertanyaan 1: dari Rp 1 modal, berapa sen yang larinya ke tiap bahan? Kita bandingkan **awal (Jan 2021)** dengan **akhir (Sep 2026)**.

In [ ]:
awal, akhir = biaya.iloc[0][BAHAN], biaya.iloc[-1][BAHAN]
selisih = akhir - awal

tabel = pd.DataFrame({
    'awal_rp': awal,
    'akhir_rp': akhir,
    'selisih_rp': selisih,
    'harga_naik_pct': 100 * (akhir / awal - 1),
    'porsi_awal_pct': 100 * awal / awal.sum(),
    'porsi_akhir_pct': 100 * akhir / akhir.sum(),
    'sumbangan_kenaikan_pct': 100 * selisih / selisih.sum(),
})
tabel.round(1)

Cara baca kolomnya:
- `porsi_*_pct`: bahan itu berapa persen dari total modal (soal ukuran).
- `harga_naik_pct`: bahan itu sendiri naik berapa persen sejak awal.
- `sumbangan_kenaikan_pct`: dari total kenaikan modal, berapa persen datang dari bahan ini (soal perubahan). Totalnya 100%. Kalau ada bahan yang turun, nilainya negatif, artinya dia justru menahan kenaikan.

In [ ]:
WARNA = {'ayam': '#993C1D', 'bawang_merah': '#C0587A', 'bawang_putih': '#C9B99A', 'minyak': '#E0A526'}
LABEL = {'ayam': 'Ayam', 'bawang_merah': 'Bawang merah', 'bawang_putih': 'Bawang putih', 'minyak': 'Minyak goreng'}

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.stackplot(biaya.index, [biaya[b] for b in BAHAN],
             labels=[LABEL[b] for b in BAHAN], colors=[WARNA[b] for b in BAHAN], alpha=0.9)
ax.set_title('Isi modal semangkuk mie ayam (tanpa mie), Rp per porsi')
ax.set_ylabel('Rp per porsi'); ax.grid(alpha=0.3)
ax.legend(loc='upper left', ncol=4, frameon=False)
plt.tight_layout(); plt.show()

## Langkah 2: Siapa yang menyumbang kenaikan?

Pertanyaan 2. Selisih modal tiap bahan dari awal ke akhir, dalam rupiah per porsi.

In [ ]:
urut = tabel['selisih_rp'].sort_values()
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.barh([LABEL[b] for b in urut.index], urut.values, color=[WARNA[b] for b in urut.index])
ax.axvline(0, color='black', linewidth=0.8)
for bar, v in zip(bars, urut.values):
    ax.text(v, bar.get_y() + bar.get_height() / 2, f'  {v:+,.0f}'.replace(',', '.'), va='center', fontsize=10)
ax.set_title('Perubahan modal per porsi, Jan 2021 ke Sep 2026 (Rp)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## Langkah 3: Apakah hasilnya cuma kebetulan titik awal?

Perbandingan di atas memakai **satu bulan** di awal dan **satu bulan** di akhir. Harga ayam dan bawang bisa loncat-loncat per bulan, jadi satu bulan bisa menyesatkan. Cek ulang pakai **rata-rata 6 bulan** di tiap ujung. Kalau urutan biang keroknya sama, kesimpulannya lebih kokoh.

In [ ]:
awal6, akhir6 = biaya[BAHAN].iloc[:6].mean(), biaya[BAHAN].iloc[-6:].mean()
cek = pd.DataFrame({'selisih_1_bulan': selisih, 'selisih_rata2_6_bulan': akhir6 - awal6})
cek['urutan_1_bulan'] = cek['selisih_1_bulan'].rank(ascending=False).astype(int)
cek['urutan_6_bulan'] = cek['selisih_rata2_6_bulan'].rank(ascending=False).astype(int)
cek.round(0)

In [ ]:
rp = lambda x: f"Rp {x:,.0f}".replace(',', '.')
koma = lambda x: f"{x:.1f}".replace('.', ',')

biang = tabel['selisih_rp'].idxmax()
total_naik = selisih.sum()
konsisten = cek['selisih_rata2_6_bulan'].idxmax() == biang

display(Markdown(f"""
## Kesimpulan kecil

- Total modal naik **{rp(total_naik)}** per porsi. **{LABEL[biang]}** menyumbang **{koma(tabel.loc[biang, 'sumbangan_kenaikan_pct'])}%** dari kenaikan itu, dan porsinya di modal sekarang **{koma(tabel.loc[biang, 'porsi_akhir_pct'])}%**.
- Harga {LABEL[biang].lower()} sendiri naik **{koma(tabel.loc[biang, 'harga_naik_pct'])}%** sejak awal.
- Cek ulang pakai rata-rata 6 bulan: biang kerok {'**tetap sama**' if konsisten else '**berbeda**, jadi hasilnya sensitif ke titik awal'}.
- Batasan: terigu belum masuk, resep adalah asumsi, dan Jan 2021 diisi dari bulan berikutnya karena data awalnya bolong.

**Next:** notebook 4, apakah modal mie ayam sama di semua kota? Kita lihat data per provinsi.
"""))